# ⚡ DM AI OS v1.5.1 — Google Colab Tesla T4 Compute Plane
### Activación en 1 Toque para Generación Remota desde iPhone

> **Instrucciones:** Pulsa el botón **Play (▶)** de la celda siguiente. Todo el entorno (GPU, ComfyUI, modelos, túnel y registro con `ai.dmorales.com.ar`) se configura de forma 100% autónoma.

In [ ]:
#@title 🚀 1. Iniciar Compute Plane (Toca Play ▶) { display-mode: "form" }

import os, sys, subprocess, time, pathlib

print("=" * 60)
print("⚡ DM AI OS v1.5.1 — Compute Plane Worker Bootstrap")
print("=" * 60)

# 1. Verificar GPU NVIDIA Tesla T4
print("🔍 [1/5] Verificando Hardware GPU...")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 2. Clonar ComfyUI en Disco efímero
print("📦 [2/5] Preparando entorno ComfyUI optimizado...")
%cd /content
if not os.path.exists("/content/ComfyUI"):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    !pip install -q -r /content/ComfyUI/requirements.txt
    !pip install -q accelerate torchvision torchaudio

# 3. Descargar checkpoint SD 1.5 FP16 (Comfy-Org Archive ~2.13 GB)
print("🧠 [3/5] Descargando checkpoint SD 1.5 FP16...")
!mkdir -p /content/ComfyUI/models/checkpoints
sd15 = pathlib.Path("/content/ComfyUI/models/checkpoints/v1-5-pruned-emaonly-fp16.safetensors")
if sd15.exists() and sd15.stat().st_size < 1_500_000_000:
    try: sd15.unlink()
    except: pass
if not sd15.exists():
    !wget -c -q --show-progress https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors -O /content/ComfyUI/models/checkpoints/v1-5-pruned-emaonly-fp16.safetensors
if sd15.exists() and sd15.stat().st_size >= 1_500_000_000:
    print(f"✅ Checkpoint OK: {sd15.name} ({round(sd15.stat().st_size / (1024**3), 2)} GB)")
else:
    print(f"❌ ERROR: Checkpoint incompleto ({sd15.stat().st_size if sd15.exists() else 0} bytes). Abortando.")
    raise SystemExit(1)

# 4. Enlazar modelos adicionales desde Google Drive si están disponibles
print("📂 [4/5] Verificando modelos adicionales...")
drive_models = "/content/drive/MyDrive/ComfyUI/models"
if os.path.exists(drive_models):
    print("💾 Google Drive detectado: Enlazando modelos...")
    !cp -rs {drive_models}/* /content/ComfyUI/models/ 2>/dev/null || true
else:
    print("   (Sin Google Drive, usando checkpoint SD 1.5)")

# 5. Descargar bootstrap actualizado y registrar worker
print("🌐 [5/5] Levantando túnel cifrado y registrando con DM AI OS...")
!wget -q https://raw.githubusercontent.com/daniel2029m-droid/dm-ai-os/main/deployment/colab_bootstrap.py -O /content/colab_bootstrap.py

os.environ['DM_AI_OS_URL'] = 'https://ai.dmorales.com.ar'
os.environ['DM_WORKER_ID'] = 'colab-comfy-primary'

# Force reload to pick up latest version from GitHub
import importlib
if 'colab_bootstrap' in sys.modules:
    importlib.reload(sys.modules['colab_bootstrap'])
else:
    import colab_bootstrap
import colab_bootstrap
colab_bootstrap.run_bootstrap()
